In [ ]:
!pip install -q pyspark

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Projet Spark COVID") \
    .master("local[*]") \
    .getOrCreate()

print("Spark fonctionne !")
print("Version :", spark.version)

Spark fonctionne !
Version : 4.0.3


In [5]:
df = spark.read.csv("full_grouped.csv", header=True, inferSchema=True)

df.show(5)

+----------+--------------+---------+------+---------+------+---------+----------+-------------+--------------------+
|      Date|Country/Region|Confirmed|Deaths|Recovered|Active|New cases|New deaths|New recovered|          WHO Region|
+----------+--------------+---------+------+---------+------+---------+----------+-------------+--------------------+
|2020-01-22|   Afghanistan|        0|     0|        0|     0|        0|         0|            0|Eastern Mediterra...|
|2020-01-22|       Albania|        0|     0|        0|     0|        0|         0|            0|              Europe|
|2020-01-22|       Algeria|        0|     0|        0|     0|        0|         0|            0|              Africa|
|2020-01-22|       Andorra|        0|     0|        0|     0|        0|         0|            0|              Europe|
|2020-01-22|        Angola|        0|     0|        0|     0|        0|         0|            0|              Africa|
+----------+--------------+---------+------+---------+--

In [6]:
from pyspark.sql.functions import col

morocco_df = df.filter(col("Country/Region") == "Morocco")

morocco_df.show(5)

+----------+--------------+---------+------+---------+------+---------+----------+-------------+--------------------+
|      Date|Country/Region|Confirmed|Deaths|Recovered|Active|New cases|New deaths|New recovered|          WHO Region|
+----------+--------------+---------+------+---------+------+---------+----------+-------------+--------------------+
|2020-01-22|       Morocco|        0|     0|        0|     0|        0|         0|            0|Eastern Mediterra...|
|2020-01-23|       Morocco|        0|     0|        0|     0|        0|         0|            0|Eastern Mediterra...|
|2020-01-24|       Morocco|        0|     0|        0|     0|        0|         0|            0|Eastern Mediterra...|
|2020-01-25|       Morocco|        0|     0|        0|     0|        0|         0|            0|Eastern Mediterra...|
|2020-01-26|       Morocco|        0|     0|        0|     0|        0|         0|            0|Eastern Mediterra...|
+----------+--------------+---------+------+---------+--

In [7]:
from pyspark.sql.functions import datediff, lit, to_date

# Convertir la colonne Date en vrai format date
morocco_df = morocco_df.withColumn(
    "Date",
    to_date(col("Date"), "yyyy-MM-dd")
)

# Ajouter la colonne days_since_start
morocco_df = morocco_df.withColumn(
    "days_since_start",
    datediff(col("Date"), lit("2020-01-22"))
)

# Afficher le résultat
morocco_df.select("Date", "days_since_start", "Confirmed").show(10)

+----------+----------------+---------+
|      Date|days_since_start|Confirmed|
+----------+----------------+---------+
|2020-01-22|               0|        0|
|2020-01-23|               1|        0|
|2020-01-24|               2|        0|
|2020-01-25|               3|        0|
|2020-01-26|               4|        0|
|2020-01-27|               5|        0|
|2020-01-28|               6|        0|
|2020-01-29|               7|        0|
|2020-01-30|               8|        0|
|2020-01-31|               9|        0|
+----------+----------------+---------+
only showing top 10 rows


In [8]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

# Fenêtre de 7 jours
window_spec = Window.orderBy("Date").rowsBetween(-6, 0)

# Créer la moyenne mobile
morocco_df = morocco_df.withColumn(
    "moving_avg_7",
    avg("New cases").over(window_spec)
)

# Afficher le résultat
morocco_df.select(
    "Date",
    "New cases",
    "moving_avg_7"
).show(15)

+----------+---------+------------+
|      Date|New cases|moving_avg_7|
+----------+---------+------------+
|2020-01-22|        0|         0.0|
|2020-01-23|        0|         0.0|
|2020-01-24|        0|         0.0|
|2020-01-25|        0|         0.0|
|2020-01-26|        0|         0.0|
|2020-01-27|        0|         0.0|
|2020-01-28|        0|         0.0|
|2020-01-29|        0|         0.0|
|2020-01-30|        0|         0.0|
|2020-01-31|        0|         0.0|
|2020-02-01|        0|         0.0|
|2020-02-02|        0|         0.0|
|2020-02-03|        0|         0.0|
|2020-02-04|        0|         0.0|
|2020-02-05|        0|         0.0|
+----------+---------+------------+
only showing top 15 rows


In [9]:
morocco_df.filter(col("New cases") > 0) \
.select("Date", "New cases", "moving_avg_7") \
.show(20)

+----------+---------+-------------------+
|      Date|New cases|       moving_avg_7|
+----------+---------+-------------------+
|2020-03-02|        1|0.14285714285714285|
|2020-03-05|        1| 0.2857142857142857|
|2020-03-10|        1| 0.2857142857142857|
|2020-03-11|        2| 0.5714285714285714|
|2020-03-12|        1| 0.5714285714285714|
|2020-03-13|        1| 0.7142857142857143|
|2020-03-14|       10|  2.142857142857143|
|2020-03-15|       11| 3.7142857142857144|
|2020-03-16|        1|  3.857142857142857|
|2020-03-17|        9|                5.0|
|2020-03-18|       11|  6.285714285714286|
|2020-03-19|       14|  8.142857142857142|
|2020-03-20|       14|               10.0|
|2020-03-21|       19| 11.285714285714286|
|2020-03-22|       19| 12.428571428571429|
|2020-03-23|       28| 16.285714285714285|
|2020-03-24|       27| 18.857142857142858|
|2020-03-25|       55| 25.142857142857142|
|2020-03-26|       50| 30.285714285714285|
|2020-03-27|       70| 38.285714285714285|
+----------

In [10]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, when, col

# Fenêtre ordonnée par date
window_spec = Window.orderBy("Date")

# Nombre de nouveaux cas du jour précédent
morocco_df = morocco_df.withColumn(
    "previous_cases",
    lag("New cases").over(window_spec)
)

# Calcul du taux de croissance
morocco_df = morocco_df.withColumn(
    "growth_rate",
    when(
        col("previous_cases") > 0,
        (col("New cases") - col("previous_cases")) / col("previous_cases")
    ).otherwise(0.0)
)

# Afficher les résultats
morocco_df.select(
    "Date",
    "New cases",
    "previous_cases",
    "growth_rate"
).show(20)

+----------+---------+--------------+-----------+
|      Date|New cases|previous_cases|growth_rate|
+----------+---------+--------------+-----------+
|2020-01-22|        0|          NULL|        0.0|
|2020-01-23|        0|             0|        0.0|
|2020-01-24|        0|             0|        0.0|
|2020-01-25|        0|             0|        0.0|
|2020-01-26|        0|             0|        0.0|
|2020-01-27|        0|             0|        0.0|
|2020-01-28|        0|             0|        0.0|
|2020-01-29|        0|             0|        0.0|
|2020-01-30|        0|             0|        0.0|
|2020-01-31|        0|             0|        0.0|
|2020-02-01|        0|             0|        0.0|
|2020-02-02|        0|             0|        0.0|
|2020-02-03|        0|             0|        0.0|
|2020-02-04|        0|             0|        0.0|
|2020-02-05|        0|             0|        0.0|
|2020-02-06|        0|             0|        0.0|
|2020-02-07|        0|             0|        0.0|


In [11]:
morocco_df.filter(col("New cases") > 0) \
.select("Date", "New cases", "previous_cases", "growth_rate") \
.show(20)

+----------+---------+--------------+--------------------+
|      Date|New cases|previous_cases|         growth_rate|
+----------+---------+--------------+--------------------+
|2020-03-02|        1|             0|                 0.0|
|2020-03-05|        1|             0|                 0.0|
|2020-03-10|        1|             0|                 0.0|
|2020-03-11|        2|             1|                 1.0|
|2020-03-12|        1|             2|                -0.5|
|2020-03-13|        1|             1|                 0.0|
|2020-03-14|       10|             1|                 9.0|
|2020-03-15|       11|            10|                 0.1|
|2020-03-16|        1|            11| -0.9090909090909091|
|2020-03-17|        9|             1|                 8.0|
|2020-03-18|       11|             9|  0.2222222222222222|
|2020-03-19|       14|            11|  0.2727272727272727|
|2020-03-20|       14|            14|                 0.0|
|2020-03-21|       19|            14| 0.3571428571428571

In [12]:
from pyspark.sql.functions import month

morocco_df = morocco_df.withColumn(
    "month",
    month(col("Date"))
)

morocco_df.select("Date", "month").show(10)

+----------+-----+
|      Date|month|
+----------+-----+
|2020-01-22|    1|
|2020-01-23|    1|
|2020-01-24|    1|
|2020-01-25|    1|
|2020-01-26|    1|
|2020-01-27|    1|
|2020-01-28|    1|
|2020-01-29|    1|
|2020-01-30|    1|
|2020-01-31|    1|
+----------+-----+
only showing top 10 rows


In [13]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "days_since_start",
        "moving_avg_7",
        "growth_rate",
        "month"
    ],
    outputCol="features"
)

data = assembler.transform(morocco_df)

data.select(
    "features",
    "Confirmed"
).show(10, truncate=False)

+-----------------+---------+
|features         |Confirmed|
+-----------------+---------+
|(4,[3],[1.0])    |0        |
|[1.0,0.0,0.0,1.0]|0        |
|[2.0,0.0,0.0,1.0]|0        |
|[3.0,0.0,0.0,1.0]|0        |
|[4.0,0.0,0.0,1.0]|0        |
|[5.0,0.0,0.0,1.0]|0        |
|[6.0,0.0,0.0,1.0]|0        |
|[7.0,0.0,0.0,1.0]|0        |
|[8.0,0.0,0.0,1.0]|0        |
|[9.0,0.0,0.0,1.0]|0        |
+-----------------+---------+
only showing top 10 rows


In [14]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Ajouter un numéro de ligne
window = Window.orderBy("Date")

data = data.withColumn("row_num", row_number().over(window))

# Nombre total de lignes
total_rows = data.count()

# Limite 80%
split_index = int(total_rows * 0.8)

# Train et Test
train = data.filter(col("row_num") <= split_index)
test = data.filter(col("row_num") > split_index)

print("Nombre de lignes total :", total_rows)
print("Train :", train.count())
print("Test :", test.count())

Nombre de lignes total : 188
Train : 150
Test : 38


In [15]:
from pyspark.ml.regression import LinearRegression

# Création du modèle
lr = LinearRegression(
    featuresCol="features",
    labelCol="Confirmed"
)

# Entraînement
lr_model = lr.fit(train)

print("Le modèle Linear Regression est entraîné avec succès.")

Le modèle Linear Regression est entraîné avec succès.


In [16]:
# Faire les prédictions sur les données de test
lr_predictions = lr_model.transform(test)

# Afficher quelques résultats
lr_predictions.select(
    "Date",
    "Confirmed",
    "prediction"
).show(10)

+----------+---------+-----------------+
|      Date|Confirmed|       prediction|
+----------+---------+-----------------+
|2020-06-20|     9839|8366.764338224775|
|2020-06-21|     9977|8412.523333552159|
|2020-06-22|    10172|8354.742362916666|
|2020-06-23|    10344|8420.491247075364|
|2020-06-24|    10907|7957.162351307773|
|2020-06-25|    11338|8087.156816103013|
|2020-06-26|    11633|8333.312659187295|
|2020-06-27|    11877|8395.743031973936|
|2020-06-28|    12052|8472.594038646092|
|2020-06-29|    12290|8468.075944574191|
+----------+---------+-----------------+
only showing top 10 rows


In [17]:
from pyspark.ml.evaluation import RegressionEvaluator

# RMSE
rmse_evaluator = RegressionEvaluator(
    labelCol="Confirmed",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = rmse_evaluator.evaluate(lr_predictions)

# R²
r2_evaluator = RegressionEvaluator(
    labelCol="Confirmed",
    predictionCol="prediction",
    metricName="r2"
)

r2 = r2_evaluator.evaluate(lr_predictions)

print("RMSE =", rmse)
print("R² =", r2)

RMSE = 6020.5358360394175
R² = -3.108121725095012


In [18]:
from pyspark.ml.regression import RandomForestRegressor

# Création du modèle
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="Confirmed",
    numTrees=50,
    seed=42
)

# Entraînement
rf_model = rf.fit(train)

print("Le modèle Random Forest est entraîné avec succès.")

Le modèle Random Forest est entraîné avec succès.


In [19]:
# Faire les prédictions
rf_predictions = rf_model.transform(test)

rf_predictions.select(
    "Date",
    "Confirmed",
    "prediction"
).show(10)

+----------+---------+-----------------+
|      Date|Confirmed|       prediction|
+----------+---------+-----------------+
|2020-06-20|     9839| 8707.09673015873|
|2020-06-21|     9977|8765.998146825397|
|2020-06-22|    10172|8823.894942279941|
|2020-06-23|    10344|8726.957215007214|
|2020-06-24|    10907|8887.138942279942|
|2020-06-25|    11338|8745.699965007214|
|2020-06-26|    11633|8754.329965007215|
|2020-06-27|    11877|8745.699965007214|
|2020-06-28|    12052|8754.329965007215|
|2020-06-29|    12290|8823.894942279941|
+----------+---------+-----------------+
only showing top 10 rows


In [20]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse = RegressionEvaluator(
    labelCol="Confirmed",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(rf_predictions)

r2 = RegressionEvaluator(
    labelCol="Confirmed",
    predictionCol="prediction",
    metricName="r2"
).evaluate(rf_predictions)

print("RMSE =", rmse)
print("R² =", r2)

RMSE = 6708.3306857414245
R² = -4.1003729956073


In [23]:
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression

# Modèle de régression linéaire
lr = LinearRegression(
    featuresCol="features",
    labelCol="Confirmed"
)

# Pipeline
pipeline = Pipeline(stages=[lr])

# Entraînement
pipeline_model = pipeline.fit(train)

print("Pipeline Spark ML créé avec succès !")

Pipeline Spark ML créé avec succès !
